<a href="https://colab.research.google.com/github/IdrisJunaidAI/efficacy-ai-learning-hub/blob/main/Malware_Analysis_Using_LLM_CSMasters_IdrisJunaid_ITA1372.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# How to build a free LLM cybersecurity lab with Google Colab and Ollama

Learn how to set up a free cybersecurity lab using Large Language Models (LLMs), Google Colab, and Ollama. Explore practical use cases for AI in cybersecurity.

## Introduction

In this notebook, we will explore the powerful combination of Large Language Models (LLMs) and cybersecurity using [Ollama](https://ollama.com/). Ollama allows us to download and run LLMs locally, providing several advantages for cybersecurity applications:

1. Data privacy: Process sensitive information without sending it to external servers.
2. Customization: Easily fine-tune models for specific cybersecurity tasks.
3. Offline capability: Perform analysis without an internet connection.
4. Cost-effective: Avoid usage fees associated with cloud-based LLM services.

We'll demonstrate two practical use-cases:
1. Generating malware information cards
2. Creating a cybersecurity news digest

While we're using Google Colab for this demonstration due to its accessibility, keep in mind some limitations:
- Runtime limits (sessions typically disconnect after 12 hours)
- Variability in GPU availability
- Potential network bandwidth constraints

**For production use, consider running this notebook on a dedicated machine or cloud instance with more stable resources.**

Let's dive in and see how we can leverage LLMs for cybersecurity tasks!

## Installation of ollama

In [1]:
!apt-get update -qq
!apt-get install -y zstd

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
The following NEW packages will be installed:
  zstd
0 upgraded, 1 newly installed, 0 to remove and 47 not upgraded.
Need to get 644 kB of archives.
After this operation, 1,845 kB of additional disk space will be used.
Get:1 http://archive.ubuntu.com/ubuntu noble-updates/main amd64 zstd amd64 1.5.5+dfsg2-2build1.1 [644 kB]
Fetched 644 kB in 1s (467 kB/s)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...


In [2]:
!curl -fsSL https://ollama.com/install.sh | sh

>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [3]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

print("Starting Ollama server...")

for _ in range(30):
    try:
        r = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=2
        )

        if r.status_code == 200:
            print("Ollama server is running.")
            break

    except requests.RequestException:
        pass

    time.sleep(1)

else:
    print("Ollama did not start.")

Starting Ollama server...
Ollama server is running.


In [4]:
!ollama list


NAME    ID    SIZE    MODIFIED 


In [9]:
MODEL = "gemma2:2b"

In [6]:
!ollama list

NAME    ID    SIZE    MODIFIED 


In [7]:
MODEL = "gemma2:2b"

In [10]:
!ollama pull gemma2:2b

In [11]:
!ollama list

NAME         ID              SIZE      MODIFIED       
gemma2:2b    8ccf136fdd52    1.6 GB    29 seconds ago    


In [12]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    timeout=120.0
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: WORKING"
        }
    ]
)

print(response.choices[0].message.content)

WORKING 



Ollama is now installed. Ollama is an open-source tool that simplifies running large language models locally.
It provides an easy way to download, run, and manage various LLMs on your machine.

If you encounter any GPU-related issues, try the following troubleshooting steps:
1. Ensure CUDA is properly installed and configured.
2. Check if the correct GPU drivers are installed.
3. Verify that Ollama has access to the GPU resources.

Now, let's start Ollama in a separate thread so we can use it throughout this notebook.

We then start ollama in a separate thread so that we can use it after

Reference: https://stackoverflow.com/a/77828874

In [13]:
import os
import asyncio
import threading

# NB: You may need to set these depending and get cuda working depending which backend you are running.
# Set environment variable for NVIDIA library
# Set environment variables for CUDA
os.environ['PATH'] += ':/usr/local/cuda/bin'
# Set LD_LIBRARY_PATH to include both /usr/lib64-nvidia and CUDA lib directories
os.environ['LD_LIBRARY_PATH'] = '/usr/lib64-nvidia:/usr/local/cuda/lib64'

async def run_process(cmd, stdout=None, stderr=None):
    print('>>> starting', *cmd)
    process = await asyncio.create_subprocess_exec(
        *cmd,
        stdout=stdout or asyncio.subprocess.PIPE,
        stderr=stderr or asyncio.subprocess.PIPE
    )

    if stdout is None and stderr is None:
        async def pipe(lines):
            async for line in lines:
                print(line.decode().strip())

        await asyncio.gather(
            pipe(process.stdout),
            pipe(process.stderr),
        )
    else:
        await process.wait()

async def start_ollama_serve():
    await run_process(['ollama', 'serve'],
                      stdout=open(os.devnull, 'w'),
                      stderr=open(os.devnull, 'w'))

def run_async_in_thread(loop, coro):
    asyncio.set_event_loop(loop)
    loop.run_until_complete(coro)
    loop.close()

# Create a new event loop that will run in a new thread
new_loop = asyncio.new_event_loop()

# Start ollama serve in a separate thread so the cell won't block execution
thread = threading.Thread(target=run_async_in_thread, args=(new_loop, start_ollama_serve()))
thread.start()

# Wait 5s for ollama to load
import time
time.sleep(5)

>>> starting ollama serve


## Playing with ollama API

In [1]:
# We first need to install dependencies
!pip install openai pydantic instructor

For this first example, we will use [gemma2](https://blog.google/technology/developers/google-gemma-2/) from Google in its 9 billion parameters version. Its currently [one of the best small](https://chat.lmsys.org/?leaderboard) (<12B parameters) and opensource model.


## Use Case I: Generating malware information cards

In cybersecurity, having quick access to malware information is crucial for threat analysis and incident response.
We'll use our LLM to generate concise "malware cards" based on malware names. This demonstrates how LLMs can be
used to quickly summarize and present complex security information.

However, it's important to note:
1. This information is generated based on the LLM's training data and may not always be up-to-date or accurate.
2. **Always verify critical information with authoritative sources before making security decisions**.
3. Be cautious about potential hallucinations, especially for lesser-known malware.

In [4]:
!pip install openai pydantic instructor jsonref

In [7]:
MODEL = "gemma2:2b"
print("Current MODEL:", MODEL)

Current MODEL: gemma2:2b


In [8]:
import subprocess
import time
import requests

ollama_process = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)

print("Starting Ollama server...")

for _ in range(30):
    try:
        r = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=2
        )

        if r.status_code == 200:
            print("Ollama server is running.")
            break

    except requests.RequestException:
        pass

    time.sleep(1)

else:
    raise RuntimeError("Ollama did not start.")

Starting Ollama server...
Ollama server is running.


In [11]:
import requests

r = requests.get(
    "http://127.0.0.1:11434/api/tags",
    timeout=5
)

print("Status:", r.status_code)
print(r.json())

Status: 200
{'models': [{'name': 'gemma2:2b', 'model': 'gemma2:2b', 'modified_at': '2026-10-04T18:54:27.253264173Z', 'size': 1629518495, 'digest': '8ccf136fdd5298f3ffe2d69862750ea7fb56555fa4d5b18c04e3fa4d82ee09d7', 'details': {'parent_model': '', 'format': 'gguf', 'family': 'gemma2', 'families': ['gemma2'], 'parameter_size': '2.6B', 'quantization_level': 'Q4_0', 'context_length': 8192, 'embedding_length': 2304}, 'capabilities': ['completion']}]}


In [13]:
from openai import OpenAI

client = OpenAI(
    base_url="http://127.0.0.1:11434/v1",
    api_key="ollama",
    timeout=120.0
)

response = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "user",
            "content": "Reply with exactly: WORKING"
        }
    ]
)

print(response.choices[0].message.content)

WORKING 



In [12]:
!ollama list

NAME         ID              SIZE      MODIFIED       
gemma2:2b    8ccf136fdd52    1.6 GB    17 minutes ago    


#### Fetch malware data

In [14]:
from openai import OpenAI
from pydantic import BaseModel
from datetime import datetime
import instructor

class Malware(BaseModel):
    name: str
    first_seen: datetime
    language: str
    architecture: str
    developer: str
    description: str

def get_malware_description(malware_name: str) -> Malware:
    openai_client = OpenAI(
        base_url="http://127.0.0.1:11434/v1",
        api_key="ollama",
        timeout=120.0,
    )

    client = instructor.from_openai(
        openai_client,
        mode=instructor.Mode.JSON,
    )

    prompt = f"""
Return a factual malware information card for {malware_name}.
Return JSON only and follow the requested schema exactly.
If a specific fact is uncertain, use "Unknown" rather than inventing details.
For first_seen, provide a valid ISO 8601 date.
"""

    return client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        response_model=Malware,
        max_retries=2,
        temperature=0,
    )

malwares = ["WannaCry", "Stuxnet", "DarkComet"]

for malware in malwares:
    try:
        resp = get_malware_description(malware)
        print(resp.model_dump_json(indent=2))
    except Exception as exc:
        print(f"Could not generate the card for {malware}: {exc}")

{
  "name": "WannaCry",
  "first_seen": "2017-05-12T00:00:00",
  "language": "Unknown",
  "architecture": "Unknown",
  "developer": "Unknown",
  "description": "A ransomware attack that spread rapidly across the globe, encrypting files and demanding payment in Bitcoin for decryption keys. It exploited a vulnerability in the Windows operating system to spread."
}
{
  "name": "Stuxnet",
  "first_seen": "2010-09-01T00:00:00Z",
  "language": "Unknown",
  "architecture": "Windows",
  "developer": "Unknown",
  "description": "A highly sophisticated computer worm that targeted the Iranian nuclear program. It was designed to sabotage the Iranian nuclear enrichment program by causing damage to centrifuges at the Natanz nuclear facility."
}
{
  "name": "DarkComet",
  "first_seen": "2017-03-01T00:00:00Z",
  "language": "English",
  "architecture": "Remote Access Trojan (RAT)",
  "developer": "Unknown",
  "description": "DarkComet is a remote access Trojan (RAT) that allows attackers to control in

## Use Case II: Building a daily cybersecurity news digest

Staying informed about the latest cybersecurity developments is essential but time-consuming. This use case
demonstrates how to leverage LLMs to create a personalized, AI-curated news digest.

Benefits:
1. Time-saving: Quickly get an overview of the most important cybersecurity news.
2. Prioritization: Focus on high-impact stories first.
3. Contextual understanding: Get AI-generated summaries and potential implications of each story.

The process involves:
1. Fetching articles from trusted RSS feeds
2. Using the LLM to rank and summarize the articles
3. Generating a comprehensive digest with actionable insights

Let's walk through each step:

In [16]:
!pip install feedparser

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.7/80.7 kB 4.7 MB/s eta 0:00:00


#### Fetching of RSS entries

In [17]:
from pydantic import BaseModel, Field
from datetime import datetime
import uuid

class Article(BaseModel):
    id: str = Field(..., description="Unique identifier for the article")
    title: str = Field(..., description="Title of the article")
    link: str = Field(..., description="URL of the article")
    published: datetime = Field(..., description="Date and time the article was published")

import feedparser
import uuid

RSS_FEEDS = [
    "https://feeds.feedburner.com/TheHackersNews",
    "https://www.bleepingcomputer.com/feed/",
]

# fetch last rss entries and return title, url and id
def fetch_rss_entries(url):
    feed = feedparser.parse(url)
    return [Article(id=uuid.uuid4().hex[:8],
                   title=entry.title,
                   link=entry.link,
                   published=datetime(*entry.published_parsed[:6]))
            for entry in feed.entries]

articles = []
for rss_feed in RSS_FEEDS:
  articles.extend(fetch_rss_entries(rss_feed))
articles

[Article(id='6ee21595', title='ShinyHunters Suspect Rey Reportedly Detained in Jordan, Helping FBI Identify Group Members', link='https://thehackernews.com/2026/10/shinyhunters-suspect-rey-reportedly.html', published=datetime.datetime(2026, 10, 4, 7, 22, 5)),
 Article(id='16a33cf9', title='China-Aligned TA419 Targets U.S. AI Policy Experts With Microsoft AitM Phishing', link='https://thehackernews.com/2026/10/china-aligned-ta419-targets-us-ai.html', published=datetime.datetime(2026, 10, 4, 7, 20, 32)),
 Article(id='8e21bb8a', title='MI5 Says China’s MSS Funded Research Involving 100+ U.K.-Linked Academics', link='https://thehackernews.com/2026/10/mi5-says-chinas-mss-funded-research.html', published=datetime.datetime(2026, 10, 3, 14, 38, 46)),
 Article(id='925cf177', title='Warlock Exploits SharePoint Flaws to Disable Security Tools and Deploy Ransomware', link='https://thehackernews.com/2026/10/warlock-exploits-sharepoint-flaws-to.html', published=datetime.datetime(2026, 10, 3, 14, 36,

#### Ranking using LLM

In [18]:
from pydantic import BaseModel, Field
from typing import List
import instructor
from openai import OpenAI
import requests


class ArticleRef(BaseModel):
    id: str
    relevance_score: int = Field(
        ...,
        ge=1,
        le=10,
        description="Relevance score from 1 to 10"
    )


class RankedCyberNews(BaseModel):
    title: str = Field(
        ...,
        max_length=100,
        description="Concise title of the cybersecurity news item"
    )

    summary: str = Field(
        ...,
        max_length=250,
        description="Brief summary of the news item"
    )

    impact_score: int = Field(
        ...,
        ge=1,
        le=10,
        description="Impact score from 1 to 10"
    )

    related_articles: List[ArticleRef] = Field(
        ...,
        min_length=1,   # FIX: Pydantic v2 uses min_length
        description="List of related article IDs with relevance scores"
    )


def rank_cyber_news(
    articles: List[Article]
) -> List[RankedCyberNews]:

    # FIX: verify Ollama before Instructor makes the request
    try:
        check = requests.get(
            "http://127.0.0.1:11434/api/tags",
            timeout=5
        )
        check.raise_for_status()

    except Exception as exc:
        raise RuntimeError(
            "Ollama is not running on port 11434. "
            "Run the Ollama server recovery cell first."
        ) from exc


    client = instructor.from_openai(
        OpenAI(
            base_url="http://127.0.0.1:11434/v1",
            api_key="ollama",
            timeout=180.0
        ),
        mode=instructor.Mode.JSON
    )


    article_data = "\n".join(
        [
            f"{a.id} - {a.title}"
            for a in articles
        ]
    )


    prompt = f"""
As a senior cybersecurity analyst, analyze the following articles.

1. Provide a concise title, maximum 100 characters.

2. Write a brief summary, maximum 250 characters.

3. Assign an impact score from 1 to 10 based on its potential consequences.

4. List related article IDs with a relevance score from 1 to 10 for each.

Consider factors such as:

The scale and severity of any breaches or vulnerabilities

The prominence of affected organizations or systems

The novelty or sophistication of attack methods

Potential long term implications for cybersecurity practices


Articles to analyze:

{article_data}


Provide your analysis in a structured format suitable for JSON parsing.
"""


    resp = client.chat.completions.create(
        model=MODEL,

        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],

        response_model=List[RankedCyberNews],

        max_retries=2,

        temperature=0
    )

    return resp


ranked_news = rank_cyber_news(articles)

ranked_news

[RankedCyberNews(title='ShinyHunters Suspect Rey Reportedly Detained in Jordan, Helping FBI Identify Group Members', summary="The FBI is working with authorities in Jordan to identify members of the ShinyHunters hacking group. The suspect, Rey, is believed to have been involved in the group's activities and is now detained.", impact_score=8, related_articles=[ArticleRef(id='6ee21595', relevance_score=9)]),
 RankedCyberNews(title='China-Aligned TA419 Targets U.S. AI Policy Experts With Microsoft AitM Phishing', summary='TA419, a Chinese-backed hacking group, has targeted U.S. AI policy experts with phishing attacks using Microsoft AitM. The attacks aim to steal sensitive information and potentially disrupt AI policy development.', impact_score=7, related_articles=[ArticleRef(id='16a33cf9', relevance_score=8)]),
 RankedCyberNews(title='MI5 Says China’s MSS Funded Research Involving 100+ U.K.-Linked Academics', summary="The UK's MI5 agency has revealed that the Chinese Ministry of State S

In [19]:
# For each subject, we will parse articles using newspaper lib and ask for an actionnable summary
!pip install newspaper4k

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.1/53.1 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.2/317.2 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.5/107.5 kB 11.9 MB/s eta 0:00:00


#### Article scraping

In [20]:
from newspaper import Article as ArticleParser

full_text = ""

for ranked_article in ranked_news:
    full_text += f"# News: {ranked_article.title}\n\n"
    full_text += f"Summary: {ranked_article.summary}\n\n"
    full_text += f"Impact Score: {ranked_article.impact_score}\n\n"

    # Select the article with the highest relevance score
    if ranked_article.related_articles:
        most_relevant_article = max(ranked_article.related_articles, key=lambda x: x.relevance_score)

        # Get article link using id
        article_link = next((a.link for a in articles if a.id == most_relevant_article.id), None)
        if article_link:
            full_text += f"# Most Relevant Article (Relevance: {most_relevant_article.relevance_score}):\n\n{article_link}\n\n"
            try:
                article_obj = ArticleParser(article_link)
                article_obj.download()
                article_obj.parse()
                full_text += f"# Text:\n\n{article_obj.text}\n\n"  # Limit to first 1000 characters
            except Exception as e:
                full_text += f"Error fetching article: {str(e)}\n\n"
    else:
        full_text += "No related articles found for this news item.\n\n"

print(f"Processed {len(ranked_news)} news items.")

Processed 12 news items.


#### Digest building

In [21]:
from openai import OpenAI

def summarize_articles(full_text: str) -> str:
    client = OpenAI(
        base_url="http://127.0.0.1:11434/v1",
        api_key="ollama",  # required, but unused
    )
    prompt = f"""
    As an expert cybersecurity analyst, create a concise yet comprehensive daily news digest in markdown format. Your task:

    1. Analyze the provided cybersecurity articles.
    2. Summarize the given stories, which are already ranked by importance.
    3. For each story, include:
       - The provided headline
       - A brief summary (2-3 sentences) highlighting key points
       - The potential impact on individuals or organizations (consider the given impact score)
       - Key details from the most relevant related article
       - Actionable advice or takeaways for readers

    4. Use bullet points for clarity and readability.
    5. Conclude with a "Bottom Line" that provides an overall assessment of the day's cybersecurity landscape.

    Guidelines:
    - Use clear, professional language avoiding unnecessary jargon.
    - Prioritize stories based on their impact scores.
    - Ensure the total digest length is between 500-750 words.
    - Use markdown formatting to enhance readability (e.g., headers, bold for emphasis).

    Articles to summarize:

    {full_text}

    Produce only the markdown-formatted digest without any additional commentary or metadata.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return resp.choices[0].message.content

digest = summarize_articles(full_text)
print(digest)

    _[session_id]."
    
"The Antino backdoor abuses the Windows Scripted Diagnostics framework to execute attacker-controlled PowerShell through legitimate Windows components," Talos said. "This can complicate behavioral attribution to the original implant, although it does not eliminate observable PowerShell, file-creation, or Registry telemetry."

# News: US sanctions Tren de Aragua gang members in ATM hacks crackdown

Summary: The US has sanctioned members of the Tren de Aragua gang for their involvement in ATM hacks. This highlights the growing threat of cybercrime and the importance of international cooperation in combating it.

Impact Score: 6

# Most Relevant Article (Relevance: 5):

https://www.bleepingcomputer.com/news/security/us-sanctions-tren-de-aragua-members-in-atm-jackpotting-crackdown/

# Text:

The U.S. Treasury Department has sanctioned eight members of the Venezuelan gang Tren de Aragua (TdA) for their role in the theft of millions of dollars in ATM jackpotting atta

#### Bonus: build your daily podcast

In this part, we will use another prompt to generate the script for a podcast about the last cybersecurity news. We will then use

In [22]:
def build_podcast(full_text: str) -> str:
    client = OpenAI(
        base_url="http://127.0.0.1:11434/v1",
        api_key="ollama",  # required, but unused
    )
    prompt = f"""
    You are an engaging cybersecurity podcast host. Create a script for a 5-minute daily news digest based on the provided articles. Your task:

    1. Start with a brief, catchy introduction.
    2. Summarize 2-3 key cybersecurity stories from the given content.
    3. For each story, provide:
       - A concise overview of the incident or development
       - Its potential impact on individuals or organizations
       - Quick, actionable advice for listeners (if applicable)
    4. Use a conversational tone, as if speaking directly to the audience.
    5. Conclude with a brief sign-off, encouraging listeners to stay vigilant.

    Guidelines:
    - Keep the total length around 600-800 words.
    - Use simple language and avoid technical jargon when possible.
    - Include relevant statistics or numbers to add context.
    - Add brief transitions between stories to maintain flow.

    Content to summarize:

    {full_text}

    Remember, create only the podcast script without any formatting characters or annotations.
    """
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    return resp.choices[0].message.content

podcast = build_podcast(full_text)
podcast

"##  Cybersecurity Spotlight:  Recent Threats and  Emerging Solutions\n\n**Introduction**\n\nWelcome to the Cybersecurity Spotlight, where we dive deep into the latest threats and developments in the digital world.  In today's episode, we'll be talking about evolving cyberattacks, international cooperation in fighting crime, and the importance of vigilance and proactive security.\n\n**The Antino Backdoor and ATM Hacks**\n\nFirst, a new report highlights a concerning development within the cybercrime landscape: malicious software, known as the Antino backdoor, is being used to execute attacker-controlled PowerShell through legitimate Windows components. This highlights the growing sophistication and complexity of cybercriminal tactics.  The report explores how this backdoor is exploited to compromise systems.\n\n**Impact on Security Professionals?** \n\nOne important takeaway is the impact on security professionals. They now face a more complex challenge, as they need to understand and 

In [24]:
!pip install unidic unidic-lite

  Using cached unidic-1.1.0.tar.gz (7.7 kB)
  Preparing metadata (setup.py) ... done
  Using cached unidic-lite-1.0.8.tar.gz (47.4 MB)
  Preparing metadata (setup.py) ... done
  Created wheel for unidic: filename=unidic-1.1.0-py3-none-any.whl size=7471 sha256=04dec63776cbd5a2d504ac2d5ceba59bc3e7111c1165bfdeddefdcba2d001a6f
  Stored in directory: /root/.cache/pip/wheels/8d/16/c0/3f6389eafde3e0dd15af1041a209556a1ffef2ed6f0eb59a36
  Created wheel for unidic-lite: filename=unidic_lite-1.0.8-py3-none-any.whl size=47658911 sha256=869d9988a7076a7a3bfc22db79bfa1e7c0f93af456c0a330765c2e944824d73b
  Stored in directory: /root/.cache/pip/wheels/93/40/f2/23f8a0da599c4174200d54d0b933aedc464a12f1061e4aabba
Successfully built unidic unidic-lite
  Attempting uninstall: wasabi
    Found existing installation: wasabi 1.1.3
    Uninstalling wasabi-1.1.3:
      Successfully uninstalled wasabi-1.1.3


In [26]:
%cd /content/MeloTTS

!pip install -U mecab-python3
!pip install -U unidic unidic-lite

!pip install -e . --no-deps

/content/MeloTTS
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 591.3/591.3 kB 17.8 MB/s eta 0:00:00
Obtaining file:///content/MeloTTS
  Preparing metadata (setup.py) ... done
  Running setup.py develop for melotts


In [27]:
import MeCab
print("MeCab imported successfully")

MeCab imported successfully


In [30]:
import sys

print("Python version:", sys.version)

try:
    import MeCab
    print("MeCab import: OK")
except Exception as e:
    print("MeCab import failed:", repr(e))

try:
    import unidic
    print("unidic import: OK")
except Exception as e:
    print("unidic import failed:", repr(e))

try:
    import melo
    print("melo import: OK")
except Exception as e:
    print("melo import failed:", repr(e))

try:
    from melo.api import TTS
    print("MeloTTS import: OK")
except Exception as e:
    print("TTS import failed:", repr(e))

Python version: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
MeCab import: OK
unidic import: OK
melo import: OK
TTS import failed: ModuleNotFoundError("No module named 'cn2an'")


In [43]:
!pip install anyascii cached_path eng_to_ipa fugashi g2p_en g2pkk \
gruut jamo langid loguru num2words pykakasi pypinyin txtsplit unidecode

In [44]:
from melo.api import TTS
print("MeloTTS imported successfully")

MeloTTS imported successfully


In [45]:
import re

clean_podcast = podcast

# Remove Markdown bold markers
clean_podcast = clean_podcast.replace("**", "")

# Remove Markdown headings and bullets
clean_podcast = re.sub(r'[#*_`]+', ' ', clean_podcast)

# Replace repeated whitespace with a single space
clean_podcast = re.sub(r'\s+', ' ', clean_podcast)

# Remove unusual characters that may confuse the phonemizer
clean_podcast = re.sub(r'[^A-Za-z0-9.,!?;:\'"\-\s]', ' ', clean_podcast)

clean_podcast = clean_podcast.strip()

print(clean_podcast[:1000])

Cybersecurity Spotlight: Recent Threats and Emerging Solutions Introduction Welcome to the Cybersecurity Spotlight, where we dive deep into the latest threats and developments in the digital world. In today's episode, we'll be talking about evolving cyberattacks, international cooperation in fighting crime, and the importance of vigilance and proactive security. The Antino Backdoor and ATM Hacks First, a new report highlights a concerning development within the cybercrime landscape: malicious software, known as the Antino backdoor, is being used to execute attacker-controlled PowerShell through legitimate Windows components. This highlights the growing sophistication and complexity of cybercriminal tactics. The report explores how this backdoor is exploited to compromise systems. Impact on Security Professionals? One important takeaway is the impact on security professionals. They now face a more complex challenge, as they need to understand and analyze the vulnerabilities that attacke

In [47]:
import nltk

nltk.download("averaged_perceptron_tagger_eng")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.


True

In [48]:
import nltk

nltk.download("averaged_perceptron_tagger_eng")
nltk.download("averaged_perceptron_tagger")
nltk.download("cmudict")
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package cmudict to /root/nltk_data...
[nltk_data]   Package cmudict is already up-to-date!
[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [49]:
from melo.api import TTS

speed = 1.1
device = "auto"

model = TTS(
    language="EN",
    device=device
)

speaker_ids = model.hps.data.spk2id

output_path = "en-default.wav"

model.tts_to_file(
    clean_podcast,
    speaker_ids["EN-Default"],
    output_path,
    speed=speed
)

print("Audio created:", output_path)

 > Text split to sentences.
Cybersecurity Spotlight: Recent Threats and Emerging Solutions Introduction Welcome to the Cybersecurity Spotlight, where we dive deep into the latest threats and developments in the digital world. In today's episode, we'll be talking about evolving cyberattacks,
international cooperation in fighting crime, and the importance of vigilance and proactive security. The Antino Backdoor and ATM Hacks First, a new report highlights a concerning development within the cybercrime landscape: malicious software, known as the Antino backdoor,
is being used to execute attacker-controlled PowerShell through legitimate Windows components. This highlights the growing sophistication and complexity of cybercriminal tactics. The report explores how this backdoor is exploited to compromise systems. Impact on Security Professionals?
One important takeaway is the impact on security professionals. They now face a more complex challenge, as they need to understand and analyze the 

  0%|          | 0/10 [00:00<?, ?it/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

[transformers] BertForMaskedLM LOAD REPORT from: bert-base-uncased
Key                         | Status     |  | 
----------------------------+------------+--+-
bert.pooler.dense.bias      | UNEXPECTED |  | 
bert.pooler.dense.weight    | UNEXPECTED |  | 
cls.seq_relationship.bias   | UNEXPECTED |  | 
cls.seq_relationship.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
100%|██████████| 10/10 [00:15<00:00,  1.58s/it]


Audio created: en-default.wav


In [50]:
from IPython.display import Audio #Import Audio method from IPython's Display Class

Audio(output_path, autoplay=False)

## Conclusion
In this notebook, we explored how Large Language Models can support cybersecurity operations using Ollama and a locally running open source model.

Two primary use cases were demonstrated:
1. generating structured malware information cards for rapid threat reference and creating an AI curated cybersecurity news digest by collecting, ranking, and
2. summarizing relevant security information.

These workflows show how LLMs can help analysts synthesize complex information, automate repetitive research tasks, prioritize important cybersecurity developments, and improve the efficiency of threat analysis and reporting.

The lab also demonstrated the importance of structured outputs, local model deployment, API based integration, and troubleshooting dependencies across the AI workflow. As an optional extension, the generated cybersecurity content was converted into a podcast style script and synthesized into playable audio using MeloTTS.

Although these capabilities can improve SOC efficiency, LLM generated information should not be treated as authoritative without verification. Malware details, threat intelligence, severity assessments, and recommended actions should be validated against trusted sources and reviewed by cybersecurity professionals before being used for operational decisions.

Overall, this lab demonstrated how LLMs can serve as valuable assistive tools within a modern Security Operations Center while reinforcing the importance of human oversight, accuracy, privacy, and responsible AI use.